# 2048 with Deep Q-Networks (Google Colab)

This notebook trains a simple Deep Q-Network (DQN) agent to play 2048. The reward is based on the score gained from each move, encouraging the agent to maximize the game score.


## Setup

Run the following cell on Google Colab to ensure dependencies are available. PyTorch is already installed on most Colab runtimes.


In [ ]:
%%capture
!pip -q install numpy matplotlib tqdm



## Imports and Utilities


In [ ]:
import math
import random
import time
from collections import deque, namedtuple
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import clear_output, display
from tqdm.auto import tqdm
# Reproducibility helpers
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {dev}")


## Board visualization

Helper functions to draw a 2048 board with colors similar to the classic game. These are used for live training previews and episode playback.


In [ ]:
# Color palette adapted from the original 2048 look
TILE_COLORS = {
    0: "#cdc1b4",
    2: "#eee4da",
    4: "#ede0c8",
    8: "#f2b179",
    16: "#f59563",
    32: "#f67c5f",
    64: "#f65e3b",
    128: "#edcf72",
    256: "#edcc61",
    512: "#edc850",
    1024: "#edc53f",
    2048: "#edc22e",
}
def render_board(board, score=0, best=None, move=None, figsize=(4.5, 4.5), return_fig=False):
    """Render a 2048 board with colored tiles and score banner."""
    size = board.shape[0]
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, size)
    ax.set_ylim(0, size)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    ax.set_facecolor('#bbada0')
    for r in range(size):
        for c in range(size):
            val = int(board[r, c])
            color = TILE_COLORS.get(val, '#3c3a32')
            rect = plt.Rectangle((c, size - r - 1), 1, 1, facecolor=color, edgecolor='#bbada0', lw=2)
            ax.add_patch(rect)
            if val:
                ax.text(
                    c + 0.5,
                    size - r - 0.5,
                    str(val),
                    ha='center',
                    va='center',
                    fontsize=16 if val < 128 else 14,
                    fontweight='bold',
                    color='#776e65' if val < 8 else '#f9f6f2',
                )
    title_parts = [f"Score: {score}"]
    if best is not None:
        title_parts.append(f"Best tile: {best}")
    if move is not None:
        title_parts.append(f"Move: {move}")
    ax.set_title(" | ".join(title_parts), fontsize=14, color='#3c3a32')
    plt.tight_layout()
    if return_fig:
        return fig
    display(fig)
    plt.close(fig)


## 2048 Environment

A minimal 2048 implementation that keeps track of the total score. The reward for the agent equals the score gained from a move (negative if the move is invalid).


In [ ]:
class Game2048:
    ACTIONS = {0: 'up', 1: 'down', 2: 'left', 3: 'right'}
    def __init__(self):
        self.size = 4
        self.reset()
    def reset(self):
        self.board = np.zeros((self.size, self.size), dtype=np.int32)
        self.score = 0
        self._add_random_tile()
        self._add_random_tile()
        return self._get_state()
    def _get_state(self):
        # Log2 scaling keeps magnitudes small; empty cells are 0
        state = np.where(self.board > 0, np.log2(self.board), 0)
        return state.astype(np.float32)
    def _add_random_tile(self):
        empty = list(zip(*np.where(self.board == 0)))
        if not empty:
            return
        r, c = random.choice(empty)
        self.board[r, c] = 4 if random.random() < 0.1 else 2
    def _compress(self, row):
        filtered = [x for x in row if x != 0]
        merged = []
        skip = False
        gain = 0
        for i in range(len(filtered)):
            if skip:
                skip = False
                continue
            if i + 1 < len(filtered) and filtered[i] == filtered[i + 1]:
                merged_val = filtered[i] * 2
                merged.append(merged_val)
                gain += merged_val
                skip = True
            else:
                merged.append(filtered[i])
        merged += [0] * (self.size - len(merged))
        return merged, gain
    def _move(self, direction):
        rotated = self.board
        if direction == 'up':
            rotated = self.board.T
        elif direction == 'down':
            rotated = np.flipud(self.board).T
        elif direction == 'right':
            rotated = np.fliplr(self.board)
        moved = np.zeros_like(rotated)
        move_gain = 0
        for i in range(self.size):
            merged_row, gain = self._compress(rotated[i])
            moved[i] = merged_row
            move_gain += gain
        if direction == 'up':
            moved = moved.T
        elif direction == 'down':
            moved = np.flipud(moved.T)
        elif direction == 'right':
            moved = np.fliplr(moved)
        return moved, move_gain
    def step(self, action):
        direction = self.ACTIONS[action]
        new_board, gain = self._move(direction)
        invalid = np.array_equal(new_board, self.board)
        if invalid:
            reward = -2.0  # small penalty for wasting a move
            done = not self.can_move()
        else:
            self.board = new_board
            self.score += gain
            self._add_random_tile()
            reward = float(gain)
            done = not self.can_move()
        return self._get_state(), reward, done, {}
    def can_move(self):
        if (self.board == 0).any():
            return True
        for r in range(self.size):
            for c in range(self.size - 1):
                if self.board[r, c] == self.board[r, c + 1]:
                    return True
        for r in range(self.size - 1):
            for c in range(self.size):
                if self.board[r, c] == self.board[r + 1, c]:
                    return True
        return False
    def render(self, move=None):
        render_board(self.board, score=self.score, best=int(self.board.max()), move=move)


## Replay Buffer


In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)



## DQN Model and Agent


In [ ]:
class DQN(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 4)
        )

    def forward(self, x):
        return self.net(x)


def epsilon_by_frame(frame_idx, eps_start=1.0, eps_final=0.05, eps_decay=8000):
    return eps_final + (eps_start - eps_final) * math.exp(-1.0 * frame_idx / eps_decay)


class DQNAgent:
    def __init__(self, state_shape, gamma=0.99, lr=1e-3, buffer_size=20000, batch_size=256, target_sync=500):
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_sync = target_sync

        input_dim = state_shape[0] * state_shape[1]
        self.policy_net = DQN(input_dim).to(dev)
        self.target_net = DQN(input_dim).to(dev)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)
        self.frame_idx = 0

    def act(self, state):
        eps = epsilon_by_frame(self.frame_idx)
        self.frame_idx += 1
        if random.random() < eps:
            return random.randrange(4)
        with torch.no_grad():
            state_v = torch.tensor(state.flatten(), device=dev).float().unsqueeze(0)
            q_values = self.policy_net(state_v)
            return int(torch.argmax(q_values, dim=1).item())

    def push(self, *args):
        self.buffer.push(*args)

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None
        transitions = self.buffer.sample(self.batch_size)
        batch = Transition(*transitions)

        state_batch = torch.tensor(np.stack(batch.state), device=dev).float()
        action_batch = torch.tensor(batch.action, device=dev).long().unsqueeze(1)
        reward_batch = torch.tensor(batch.reward, device=dev).float()
        next_state_batch = torch.tensor(np.stack(batch.next_state), device=dev).float()
        done_batch = torch.tensor(batch.done, device=dev).float()

        q_values = self.policy_net(state_batch.view(self.batch_size, -1)).gather(1, action_batch).squeeze(1)

        with torch.no_grad():
            next_q = self.target_net(next_state_batch.view(self.batch_size, -1)).max(1)[0]
            target = reward_batch + self.gamma * next_q * (1 - done_batch)

        loss = nn.functional.smooth_l1_loss(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 5.0)
        self.optimizer.step()

        if self.frame_idx % self.target_sync == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        return loss.item()



## Training Loop

The default configuration keeps the runtime modest for demonstration. Increase `num_episodes` or tweak the hyperparameters for better play strength when running on Colab with a GPU.


In [ ]:
num_episodes = 300
max_steps = 800
# Toggle to see a live view of the agent during training (may slow things down)
enable_live_view = True
live_every = 25  # visualize every N episodes
env = Game2048()
agent = DQNAgent(env.board.shape)
reward_history = []
loss_history = []
best_score = 0
live_display = None
for episode in tqdm(range(num_episodes), desc="Training"):
    state = env.reset()
    episode_reward = 0
    for step in range(max_steps):
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        episode_reward += reward
        agent.push(state, action, reward, next_state, done)
        loss = agent.update()
        if loss is not None:
            loss_history.append(loss)
        if enable_live_view and (episode % live_every == 0) and step % 3 == 0:
            fig = render_board(env.board, score=env.score, best=int(env.board.max()), move=Game2048.ACTIONS[action], return_fig=True)
            if live_display is None:
                live_display = display(fig, display_id=True)
            else:
                live_display.update(fig)
            print(f"Episode {episode+1}/{num_episodes} | Step {step+1} | Epsilon {epsilon_by_frame(agent.frame_idx):.3f}")
            time.sleep(0.1)
        state = next_state
        if done:
            best_score = max(best_score, env.score)
            break
    reward_history.append(episode_reward)
print(f"Training complete. Best observed score: {best_score}")


## Evaluation

Run a few greedy games to see how the trained agent performs.


In [ ]:
def run_episode(env, agent, render=False):
    state = env.reset()
    total_reward = 0
    with torch.no_grad():
        while True:
            action = agent.policy_net(torch.tensor(state.flatten(), device=dev).float().unsqueeze(0)).argmax(1).item()
            next_state, reward, done, _ = env.step(action)
            total_reward += reward
            state = next_state
            if render:
                clear_output(wait=True)
                render_board(env.board, score=env.score, best=int(env.board.max()), move=Game2048.ACTIONS[action])
                time.sleep(0.15)
            if done:
                return total_reward, env.score
scores = []
for _ in range(5):
    r, score = run_episode(Game2048(), agent, render=False)
    scores.append(score)
print(f"Average evaluation score over 5 games: {np.mean(scores):.1f}")


### Visualize a trained agent

Run a single episode with real-time board updates to watch the policy play.


In [ ]:
def animate_episode(agent, delay=0.2, max_steps=800):
    env = Game2048()
    state = env.reset()
    with torch.no_grad():
        for step in range(max_steps):
            action = agent.policy_net(torch.tensor(state.flatten(), device=dev).float().unsqueeze(0)).argmax(1).item()
            next_state, reward, done, _ = env.step(action)
            clear_output(wait=True)
            render_board(env.board, score=env.score, best=int(env.board.max()), move=Game2048.ACTIONS[action])
            time.sleep(delay)
            state = next_state
            if done:
                print(f"Final score: {env.score}")
                break
# animate_episode(agent, delay=0.25)  # Uncomment after training to watch the agent


## Play Manually

After training, you can play manually to compare against the learned policy. Use WASD keys to move.


In [ ]:
def manual_play():
    env = Game2048()
    key_to_action = {'w':0, 's':1, 'a':2, 'd':3}
    state = env.reset()
    while True:
        clear_output(wait=True)
        render_board(env.board, score=env.score, best=int(env.board.max()))
        move = input("Move (w/a/s/d, q to quit): ").strip().lower()
        if move == 'q':
            break
        if move not in key_to_action:
            print("Invalid key, try again.")
            continue
        _, reward, done, _ = env.step(key_to_action[move])
        print(f"Reward: {reward}")
        if done:
            clear_output(wait=True)
            render_board(env.board, score=env.score, best=int(env.board.max()))
            print("Game over!")
            break
# manual_play()  # Uncomment to play in an interactive environment
